In [ ]:
import os
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, average_precision_score, f1_score, precision_recall_curve

# ====== CONFIG ======
IN_FILE = "merged_id_text_label.csv"  # <-- cambia se diverso
MODEL_OUT = "catboost_td_distilled-v2.cbm"
META_OUT = "catboost_td_distilled_meta-v2.json"
TEXT_COL = "orig_text"
TARGET_COL = "logit_td"  # distillazione fedele
LABEL_COL = "label"      # opzionale per metriche classificazione
RANDOM_STATE = 42
TEST_SIZE = 0.2

# ====== LOAD ======
df = pd.read_csv(IN_FILE)
df.columns = [c.strip() for c in df.columns]

# sanity checks
required = [TEXT_COL, TARGET_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Mancano colonne richieste: {missing}. Colonne disponibili: {df.columns.tolist()}")

# pulizia base
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df[df[TEXT_COL].str.len() > 0].copy()

# target
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()

# ====== FEATURE COLUMNS ======
num_cols = []
feature_cols = [TEXT_COL]

# ====== SPLIT ======
train_df, valid_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_COL] if LABEL_COL in df.columns else None
)

train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df[TARGET_COL],
    text_features=[TEXT_COL]
)
valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df[TARGET_COL],
    text_features=[TEXT_COL]
)

# ====== TRAIN ======
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=200,
    verbose=200
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# ====== EVAL (distillation fidelity) ======
pred_logit = model.predict(valid_pool)
rmse = np.sqrt(mean_squared_error(valid_df[TARGET_COL].values, pred_logit))
print(f"\nRMSE(logit) valid: {rmse:.6f}")

# ====== EVAL (optional classification vs label) ======
# sigmoid
p_hat = 1.0 / (1.0 + np.exp(-pred_logit))
if LABEL_COL in valid_df.columns and valid_df[LABEL_COL].notna().any():
    y_true = valid_df[LABEL_COL].astype(int).values
    # PR-AUC
    pr_auc = average_precision_score(y_true, p_hat)
    print(f"PR-AUC valid (vs label): {pr_auc:.6f}")
    # soglia ottima per F1 su validation
    precisions, recalls, thresholds = precision_recall_curve(y_true, p_hat)
    f1s = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[max(best_idx - 1, 0)]) if len(thresholds) else 0.5
    y_pred = (p_hat >= best_thr).astype(int)
    f1 = f1_score(y_true, y_pred)
    print(f"Best threshold (F1): {best_thr:.6f} | F1 valid: {f1:.6f}")
else:
    pr_auc = None
    best_thr = None
    f1 = None
    print("Label non presente (o vuota) -> salto metriche classificazione.")

# ====== SAVE ======
model.save_model(MODEL_OUT)
meta = {
    "input_file": IN_FILE,
    "text_col": TEXT_COL,
    "target_col": TARGET_COL,
    "label_col": LABEL_COL if LABEL_COL in df.columns else None,
    "num_cols": num_cols,
    "feature_cols": feature_cols,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "rmse_valid_logit": rmse,
    "pr_auc_valid": pr_auc,
    "best_threshold_valid": best_thr,
    "f1_valid": f1,
    "model_file": MODEL_OUT
}
with open(META_OUT, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("\nSalvato modello:", MODEL_OUT)
print("Salvati metadata:", META_OUT)
print("Num features:", len(feature_cols), "| Numeric:", len(num_cols))


In [ ]:
# ====== OPTION A: CHEAP ONLY (no n_tokens_*) ======
num_cols_A = [c for c in df_feats.columns if c.startswith(("n_", "pct_", "kw_", "has_"))]
num_cols_A = [c for c in num_cols_A if not c.startswith("n_tokens_")]  # safety
feature_cols_A = [TEXT_COL] + num_cols_A

train_eval_save(
    df_feats_local=df_feats,
    feature_cols=feature_cols_A,
    model_out="catboost_td_distilled_A_cheap_only.cbm",
    meta_out="catboost_td_distilled_A_cheap_only.json"
)

In [ ]:
# ====== OPTION B: CHEAP + n_tokens_* via tokenizer ======
from transformers import AutoTokenizer

TOKENIZER_NAME = "karths/binary_classification_train_TD"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
SPECIAL = {"<s>", "</s>", "<pad>"}

def is_noise_token(tok: str) -> bool:
    if tok in SPECIAL:
        return True
    if tok.strip() == "" or set(tok) == {"Ġ"}:
        return True
    return False

def add_n_tokens(text: str) -> dict:
    enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, add_special_tokens=True)
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    n_all = len(toks)
    n_used = sum(0 if is_noise_token(t) else 1 for t in toks)
    return {"n_tokens_all": n_all, "n_tokens_used": n_used}

# IMPORTANT: riparti pulito per evitare colonne duplicate
dfB = df_feats.copy()
dfB = dfB.drop(columns=[c for c in ["n_tokens_all","n_tokens_used"] if c in dfB.columns], errors="ignore")
ntok = dfB[TEXT_COL].apply(add_n_tokens).apply(pd.Series)
dfB = pd.concat([dfB, ntok], axis=1)

# rimuovi colonne duplicate (ulteriore safety)
dfB = dfB.loc[:, ~dfB.columns.duplicated()]

num_cols_B = [c for c in dfB.columns if c.startswith(("n_", "pct_", "kw_", "has_"))]
seen = set()
num_cols_B = [c for c in num_cols_B if (c not in seen and not seen.add(c))]
feature_cols_B = [TEXT_COL] + num_cols_B

assert dfB.columns.duplicated().sum() == 0, "dfB ha colonne duplicate"
assert len(feature_cols_B) == len(set(feature_cols_B)), "feature_cols_B duplicata"

train_eval_save(
    df_feats_local=dfB,
    feature_cols=feature_cols_B,
    model_out="catboost_td_distilled_B_cheap_plus_ntokens.cbm",
    meta_out="catboost_td_distilled_B_cheap_plus_ntokens.json"
)